# 07. Оценка Chronos-Bolt: ряды МО и национальные ряды

Прогнозы Chronos считаются в Colab (`colab/04_…`, `colab/06_…`) и кладутся в `results/`.
Здесь они сравниваются с Prophet и бейзлайнами на тех же отсечках.

In [ ]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error as mae, r2_score as r2
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES, RAW = ROOT / "results", ROOT / "data" / "raw"

## 1. Chronos на рядах МО (те же отсечки и выборка, что в 02_backtest)

In [ ]:
c = pd.read_parquet(RES / "chronos_forecasts.parquet")
c["h"] = (c.ds.dt.year - c.origin.dt.year) * 12 + c.ds.dt.month - c.origin.dt.month
rows = []
for h in [1, 3, 6, 12]:
    d = pd.read_parquet(RES / f"predictions_h{h}.parquet").drop(columns=["chronos"], errors="ignore")
    d = d.merge(c[c.h == h][["mo_id", "origin", "yhat"]].rename(columns={"yhat": "chronos"}), on=["mo_id", "origin"], how="left")
    if "lgbm_pooled" in d:
        d["ens_snt_lgbm"] = d[["seasonal_naive_trend", "lgbm_pooled"]].mean(axis=1, skipna=False)
    s = d[d.prophet.notna()].dropna(subset=["y_true"])
    for m in ["naive", "seasonal_naive_trend", "prophet", "lgbm_pooled", "ens_snt_lgbm", "chronos"]:
        if m in s and s[m].notna().any():
            x = s.dropna(subset=[m]); rows.append({"h": h, "модель": m, "MAE": mae(x.y_true, x[m]), "R2": r2(x.y_true, x[m])})
t = pd.DataFrame(rows); t.to_csv(RES / "forecast_metrics_with_chronos.csv", index=False)
print(t.pivot(index="модель", columns="h", values="MAE").round(0).to_string())

## 2. Chronos на национальных рядах СберИндекса (2018–2026)

In [ ]:
from prophet import Prophet
cn = pd.read_parquet(RES / "chronos_national.parquet")
sp = pd.read_parquet(RAW / "consumer-spending_ru_1791179920805.parquet"); sp["date"] = pd.to_datetime(sp.period)
y = sp.pivot_table(index="date", columns="type", values="value").sort_index()
rows = []
for T in sorted(cn.origin.unique()):
    for s in y.columns:
        hist = y.loc[y.index <= T, s].dropna()
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False, seasonality_mode="multiplicative")
        m.fit(pd.DataFrame({"ds": hist.index, "y": hist.values}))
        fut = pd.DataFrame({"ds": pd.date_range(T + pd.DateOffset(months=1), periods=12, freq="MS")})
        for ds, ph in zip(fut.ds, m.predict(fut).yhat):
            ly, lyT = ds - pd.DateOffset(months=12), T - pd.DateOffset(months=12)
            rows.append({"series": s, "origin": T, "ds": ds, "h": (ds.year - T.year) * 12 + ds.month - T.month,
                         "prophet": ph, "naive": y.at[T, s], "seasonal_naive_trend": y.at[ly, s] * y.at[T, s] / y.at[lyT, s]})
b = pd.DataFrame(rows).merge(cn[["series", "origin", "ds", "yhat", "q10", "q90"]].rename(columns={"yhat": "chronos"}),
                             on=["series", "origin", "ds"])
b["y_true"] = [y.at[d, s] if d in y.index else np.nan for d, s in zip(b.ds, b.series)]
b = b.dropna(subset=["y_true"]); b["ens_chronos_snt"] = b[["chronos", "seasonal_naive_trend"]].mean(axis=1)
res = []
for h in [1, 3, 6, 12]:
    x = b[b.h == h]
    for m in ["naive", "seasonal_naive_trend", "prophet", "chronos", "ens_chronos_snt"]:
        res.append({"h": h, "модель": m, "MAPE_%": (np.abs(x[m] / x.y_true - 1)).mean() * 100, "MAE_млрд": np.abs(x[m] - x.y_true).mean()})
    print(f"h={h}: покрытие интервала Chronos 10–90%: {((x.y_true >= x.q10) & (x.y_true <= x.q90)).mean():.0%}")
r = pd.DataFrame(res); r.to_csv(RES / "national_forecast_metrics.csv", index=False)
print(r.pivot(index="модель", columns="h", values="MAPE_%").round(2).to_string())